# Comparing the speeds of `rotation_method='scipy'` and `rotation_method='qr'`
`neuromodes` contains a variety of eigenmode-related functions, including [eigenstrapping](https://direct.mit.edu/imag/article/doi/10.1162/IMAG.a.71/131445/Generation-of-surrogate-brain-maps-preserving). We have re-implemented the functionality here, with some changes to simplify installation, increase speed, process multiple maps concurrently, and facilitate reproducibility. Nonetheless, under a specific configuration, the function can exactly match the default usage of the implementation (i.e. generating the same nulls when the corresponding seed is set). In particular, we implement 2 different methods for generating rotation matrices to spin the eigenbasis: `rotation_matrix='scipy'` (which closely matches the original), and `rotation_matrix='qr'` (which is faster and more stable). 

For more details on how to use `neuromodes.eigenstrap(rotation_matrix='scipy'` to exactly match `SurfaceEigenstrapping`, see [this notebook](https://neuromodes.readthedocs.io/en/latest/validation/eigenstrapping_match_orig.html). 

Here, we compare the speeds of the `rotation_method='scipy'` and `rotation_method='qr'` methods. 


In [1]:
import time
import numpy as np
from neuromodes.eigen import EigenSolver
from neuromodes.io import fetch_surf

In [ ]:
# Params
density = '32k'

# These parameters should be tuples that specify what kinds of comparison to run
# Note that all combinations of these parameters will be run - don't make them too big!
n_groups = (10,20)
n_maps = (10,100)
n_nulls = (10,100)

In [3]:
# Initialise mesh, solve eigenmodes, and generate random data
mesh, medmask = fetch_surf(density=density)
print(f"\nInitilising mesh with {density} vertices and {max(n_groups)**2} modes.")
tic = time.time()
solver = EigenSolver(mesh, mask=medmask).solve(n_modes=max(n_groups)**2)
print(f"Time to solve eigenmodes: {time.time() - tic:.5f} seconds.\n")

test_data = np.random.default_rng().normal(loc=1, size=(solver.n_verts, max(n_maps)))  # random normal data, non-zero mean


Initilising mesh with 32k vertices and 400 modes.
Time to solve eigenmodes: 58.69682 seconds.



## There is a substantial difference when many nulls are requested
Changing the number of nulls shows how long is spent on the actual rotation (which is the main part that differs between the two methods)

In [4]:
# Profile the two methods, using a large number of nulls
# (Note that this is more than what is reqested above)

for rotation_method in ['qr', 'scipy']:
    n_null = min(n_maps) * max(n_nulls)
    n_map = min(n_maps)
    n_group = min(n_groups)

    tic = time.time()
    solver.eigenstrap(test_data[:,:n_map], n_nulls=n_null, n_groups=n_group, rotation_method=rotation_method)
    toc = time.time()
    print(f"Time to generate {n_null} nulls for {n_map} maps with {n_group**2} modes using {rotation_method}: {toc - tic:.5f} seconds.")

Time to generate 1000 nulls for 10 maps with 100 modes using qr: 3.06338 seconds.
Time to generate 1000 nulls for 10 maps with 100 modes using scipy: 7.94019 seconds.


# The difference is not as great when many maps are requested
Changing the number of maps shows how long is spent on matrix operations (allocating space, matrix multiplication, etc)

In [5]:
# Profile the two methods, using a large number of maps
# (Note that this is more than what is reqested above)

for rotation_method in ['qr', 'scipy']:
    n_null = min(n_nulls)
    n_map = min(n_nulls) * max(n_maps)
    n_group = min(n_groups)

    tic = time.time()
    solver.eigenstrap(test_data[:,:n_map], n_nulls=n_null, n_groups=n_group, rotation_method=rotation_method)
    toc = time.time()
    print(f"Time to generate {n_null} nulls for {n_map} maps with {n_group**2} modes using {rotation_method}: {toc - tic:.5f} seconds.")

Time to generate 10 nulls for 1000 maps with 100 modes using qr: 0.33080 seconds.
Time to generate 10 nulls for 1000 maps with 100 modes using scipy: 0.36277 seconds.


# Broad comparison of all different parameter combinations

In [6]:
# Test all combinations of n_maps/n_groups/n_nulls

s = 12 # for spacing only
for rotation_method in ['qr', 'scipy']:
    print(f"===== ROTATION METHOD: {rotation_method} =====")
    for n_group in n_groups:
        print(f"\n{n_group**2} modes:")         # all these print statements are formatting output
        print(f"{'Maps | Nulls'}", end="")
        for n_null in n_nulls:
            print(f"{n_null:>{s}}", end="")
        print()
        for n_map in n_maps:
            print(f"{n_map:<{s}}", end="")
            for n_null in n_nulls:
                tic = time.time()               # this is where the computation (and timing) is
                solver.eigenstrap(test_data[:,:n_map], n_nulls=n_null, n_groups=n_group, rotation_method=rotation_method)
                toc = time.time()
                print(f"{toc - tic:>{s}.6f}", end="")
            print()
    print()

===== ROTATION METHOD: qr =====

100 modes:
Maps | Nulls          10         100
10              0.159115    0.415282
100             0.243200    1.842175

400 modes:
Maps | Nulls          10         100
10              0.319968    0.888371
100             0.814210    5.966942

===== ROTATION METHOD: scipy =====

100 modes:
Maps | Nulls          10         100
10              0.154105    0.864693
100             0.330239    2.617555

400 modes:
Maps | Nulls          10         100
10              0.581144    3.378814
100             3.830924    7.719851

